# VLM-DENTAL — Stage 1: Supervised Fine-Tuning (SFT)

This notebook trains **Qwen/Qwen3.5-9B** using native **BF16 LoRA** ($r=32, \alpha=64$) on verified expert dental clinical traces across 3 staged curriculum milestones.

### Core Architectural & Clinical Invariants:
- **3-Stage Curriculum with Negative Controls Calibration**:
  - **Stage 1a (`dentex_alone`)**: 678 DENTEX disease traces + 27 healthy DENTEX negative controls $\rightarrow$ `qwen3_5_9b_sft_{track}_dentex`.
  - **Stage 1b (`dentex_tufts_overlap`)**: DENTEX + Tufts overlapping disease (caries, periapical) + Healthy controls (DENTEX + Tufts) $\rightarrow$ `qwen3_5_9b_sft_{track}_dentex_tufts_overlap`.
  - **Stage 1c (`multicohort_all`)**: DENTEX (4 findings) + Tufts Full (all 4 findings on their own) + Full healthy controls $\rightarrow$ `qwen3_5_9b_sft_{track}_multicohort_all`.
- **Multimodal Vision Projector LoRA**: Adapts `merger.mlp.0` and `merger.mlp.2` alongside LLM attention/MLP projections, specializing radiographic feature projection without catastrophic forgetting of base ViT representations.
- **Native Image Resolutions**: Native $2:1$ panoramic X-ray aspect ratios and resolutions preserved without downsampling, leveraging the 128 GB TPU v5e-8 HBM.
- **Hardware Optimization**: Multi-Core Distributed Execution on **Google Cloud TPU v5e-8** (`xmp.spawn` 8-way data-parallel with cross-replica gradient synchronization) and multi-GPU Accelerate.
- **Strict Track Segregation**:
  - **Track A (`with_tools`)**: Multi-turn agent trained on real workstation tool-use traces (8 tools).
  - **Track B (`no_tools`)**: Single-turn direct radiologist trained on tool-free Chain-of-Thought (CoT) traces.
- **Cosine Warmup & Gradient Clipping**: 5% linear warmup, cosine decay to $1\times 10^{-6}$, and gradient norm clipping at $1.0$.
- **Validation Checkpointing**: 5% held-out validation split with `best_adapter` retention.
- **Hugging Face Hub Checkpoint Sync**: Checkpoints (~760 MB LoRA + optimizer) are pushed every 25 steps to survive Kaggle 9-hour session limits and enable seamless multi-account resume.

## 1. Platform Detection & Shallow Repository Clone

Detects runtime environment (Kaggle vs Colab vs Local) and performs a shallow clone (`--depth 1`) to eliminate history download overhead.

In [ ]:
import os
import sys
from pathlib import Path

# 1. Detect platform environment
IS_KAGGLE = os.path.exists("/kaggle") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ
PLATFORM_NAME = "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "Local PC / Server")
print(f"[PLATFORM] Detected Runtime Environment: {PLATFORM_NAME}")

# 2. Shallow clone repository (--depth 1) to conserve bandwidth, disk space, and time
REPO_NAME = "VLM-DENTAL"
REPO_URL = "https://github.com/rezaxr14/VLM-DENTAL.git"

if not os.path.exists(REPO_NAME) and not os.path.exists("dental_agent"):
    print(f"[CLONE] Performing shallow clone (--depth 1) of {REPO_URL}...")
    !git clone --depth 1 {REPO_URL}
    %cd {REPO_NAME}
elif os.path.exists(REPO_NAME):
    %cd {REPO_NAME}
    print(f"[WORKSPACE] Switched directory to {os.getcwd()}")
else:
    print(f"[WORKSPACE] Already inside repository root: {os.getcwd()}")

## 2. Hardened Dependency Installation & Environment Setup

Installs core project dependencies before calling any hardware-specific libraries. Configures `PJRT_DEVICE=TPU` for Kaggle TPU v5e-8.

In [ ]:
# Install VLM-DENTAL in editable mode and essential PEFT / training packages
# Upgrading transformers to >=5.0.0 is strictly required for Qwen3.5 architecture support
# Uninstall incompatible pre-installed torchao (0.10.0) to prevent PEFT lora dispatcher crash
!pip uninstall -y torchao
!pip install -q -U "transformers>=5.0.0" accelerate peft trl datasets huggingface_hub ultralytics python-dotenv qwen-vl-utils tabulate
!pip install -q --no-deps -e .


## 3. Hardware & Cloud TPU v5e-8 Topology Detection

Initializes PyTorch/XLA on TPU v5e-8 (reporting 8-way distributed TPU device count and chip topology) or reports CUDA GPUs.

In [ ]:
import os
import torch

IS_TPU = False
DEVICE_STR = "cpu"

# Detect TPU availability without initializing the PJRT hardware client in the
# notebook kernel. Calling xm.xla_device() in the notebook kernel locks /dev/vfio/*
# hardware devices, which prevents child training scripts (!python scripts/train_sft.py)
# from spawning multi-core workers with 'open(/dev/vfio/1): Device or resource busy'.
if os.path.exists("/dev/vfio") or os.environ.get("PJRT_DEVICE") == "TPU" or "COLAB_TPU_ADDR" in os.environ:
    try:
        import torch_xla
        IS_TPU = True
        DEVICE_STR = "Cloud TPU (8 cores / chips via PyTorch/XLA)"
        print(f"[HARDWARE] SUCCESS: Detected {DEVICE_STR}")
        print(f"[HARDWARE] 8-Way PyTorch/XLA FSDP (xmp.spawn) ready across 128 GB total HBM (2.3 GB / core).")
        
        # TPU attention fusions are managed natively via PyTorch SDPA (--attn-implementation sdpa)
        print('[HARDWARE] Attention optimization configured via native PyTorch SDPA on TPU.')
        # TPU attention fusions are managed natively via PyTorch SDPA (--attn-implementation sdpa)
        print('[HARDWARE] Attention optimization configured via native PyTorch SDPA on TPU.')
    except ImportError:
        pass

if not IS_TPU:
    if torch.cuda.is_available():
        gpu_count = torch.cuda.device_count()
        gpu_name = torch.cuda.get_device_name(0)
        DEVICE_STR = f"CUDA ({gpu_count}x {gpu_name})"
        print(f"[HARDWARE] SUCCESS: Detected {DEVICE_STR}")
    else:
        print("[HARDWARE] Running on CPU / Standard environment")

print(f"[FRAMEWORK] PyTorch: {torch.__version__}")

## 4. Secrets Diagnostics, Hub Auth & Verified Traces Sync

Audits environment variables from `.env`, Kaggle Secrets Vault, and Colab Userdata, authenticates with Hugging Face Hub, and synchronizes verified clinical trace splits.

In [ ]:
from dotenv import load_dotenv
from huggingface_hub import login, snapshot_download
from tabulate import tabulate

# 1. Load local .env if present
load_dotenv()

# Helper for Kaggle Secrets / Colab Userdata ingestion
def get_secret_multisource(key: str, default: str | None = None) -> tuple[str | None, str]:
    val = os.environ.get(key)
    if val:
        return val, ".env / OS Env"

    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            sec = UserSecretsClient().get_secret(key)
            if sec:
                os.environ[key] = sec
                return sec, "Kaggle Secrets"
        except Exception:
            pass

    if IS_COLAB:
        try:
            from google.colab import userdata
            sec = userdata.get(key)
            if sec:
                os.environ[key] = sec
                return sec, "Colab Userdata"
        except Exception:
            pass

    if default is not None:
        os.environ[key] = default
        return default, "Default Fallback"

    return None, "Not Set"

def mask_secret(val: str | None, is_secret: bool = True) -> str:
    if not val:
        return "---"
    if not is_secret:
        return val
    if len(val) <= 8:
        return "********"
    return f"{val[:4]}...{val[-4:]}"

# Audit list of project environment variables & tokens
TOKEN_SPECS = [
    ("HF_TOKEN", True, None),
    ("HF_ARTIFACT_REPO", False, "Reza-Nadimi/vlm-dental-models"),
    ("HF_TRACES_REPO", False, "Reza-Nadimi/vlm-dental-traces"),
    ("DENTEX_IMAGES_REPO", False, "Reza-Nadimi/dentex-train-images"),
    ("TUFTS_IMAGES_REPO", False, "Reza-Nadimi/tufts-train-images"),
    ("OPENAI_API_KEY", True, None),
    ("ANTHROPIC_API_KEY", True, None),
    ("GEMINI_API_KEY", True, None),
    ("GROQ_API_KEY", True, None),
    ("NVIDIA_API_KEY", True, None),
    ("WANDB_API_KEY", True, None),
    ("DENTAL_AGENT_DATA_DIR", False, "data"),
]

status_table = []
for key, is_sec, def_val in TOKEN_SPECS:
    val, source = get_secret_multisource(key, def_val)
    status_str = "[SET]" if val else "[NOT SET]"
    status_table.append([key, status_str, source, mask_secret(val, is_sec)])

print("=" * 80)
print("VLM-DENTAL: ENVIRONMENT & AUTHENTICATION DIAGNOSTICS")
print("=" * 80)
print(tabulate(status_table, headers=["Environment Variable", "Status", "Detection Source", "Configured / Masked Value"], tablefmt="fancy_grid"))

# 2. Authenticate Hugging Face Hub
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    try:
        login(token=hf_token, add_to_git_credential=True)
        print("\n[AUTH] Successfully authenticated with Hugging Face Hub.")
    except Exception as e:
        print(f"\n[AUTH WARNING] Hugging Face login failed: {e}")
else:
    print("\n[AUTH WARNING] No HF_TOKEN detected. Checkpoint upload will be disabled unless logged in interactively:")
    login()

# 3. Synchronize verified clinical trace splits from Hugging Face Hub
print("\n[SYNC] Synchronizing verified clinical trace splits from Hugging Face...")
!python scripts/sync_traces_hf.py --download

# Verify that token length manifest was downloaded alongside traces
manifest_p = Path("data/traces/trace_token_lengths.json")
if manifest_p.exists():
    print(f"[SYNC] trace_token_lengths.json verified ({manifest_p.stat().st_size / 1024:.1f} KB).")
else:
    print("[SYNC NOTICE] trace_token_lengths.json not yet found on Hub. Traces will load without manifest masking until uploaded.")

# =========================================================================
# 4. PANORAMIC RADIOGRAPHIC IMAGES INGESTION (DUAL-PATH ARCHITECTURE)
# =========================================================================
# Path A (Attached Kaggle Dataset): Auto-detects /kaggle/input/dentex-panoramic
#        and /kaggle/input/tufts-panoramic if attached to this session.
#        (0 download time, 0 MB disk space used from your 40 GB workspace quota).
# Path B (Traditional Hugging Face Hub): Downloads directly from Hugging Face Hub
#        ('Reza-Nadimi/dentex-train-images' and 'Reza-Nadimi/tufts-train-images').
#
# TOGGLE: Set FORCE_HF_DATASET_DOWNLOAD = True to force remote Hugging Face
# download even when Kaggle inputs are attached.
FORCE_HF_DATASET_DOWNLOAD = False

# --- A. DENTEX Ingestion ---
dentex_local = Path("data/dentex")
dentex_local.mkdir(parents=True, exist_ok=True)
kaggle_dentex_candidates = [
    Path("/kaggle/input/datasets/rezanadimikj/dentex-panoramic"),
    Path("/kaggle/input/dentex-panoramic"),
    Path("/kaggle/input/dentex"),
]
dentex_mounted = False

if not FORCE_HF_DATASET_DOWNLOAD:
    for cand in kaggle_dentex_candidates:
        if cand.exists():
            target_img_dir = cand / "images" if (cand / "images").exists() else cand
            link_target = dentex_local / "images"
            if not link_target.exists():
                try:
                    link_target.symlink_to(target_img_dir)
                except Exception:
                    pass
            print(f"[DATASET] Attached Kaggle DENTEX detected at {cand} (Skipping remote download).")
            dentex_mounted = True
            break

if not dentex_mounted:
    dentex_repo = os.environ.get("DENTEX_IMAGES_REPO", "Reza-Nadimi/dentex-train-images")
    has_dentex = (dentex_local / "images").exists() or (dentex_local / "train_images").exists() or Path("data/images").exists()
    if dentex_repo and not has_dentex:
        print(f"\n[SYNC] Downloading DENTEX panoramic images from Hugging Face ({dentex_repo})...")
        try:
            snapshot_download(
                repo_id=dentex_repo,
                repo_type="dataset",
                local_dir=str(dentex_local),
                token=hf_token,
            )
            print("[SYNC] DENTEX images ready.")
        except Exception as e:
            print(f"[SYNC WARNING] DENTEX images download failed: {e}")
    else:
        print("\n[SYNC] DENTEX panoramic images verified on local disk.")

# --- B. Tufts Ingestion ---
tufts_local = Path("data/tufts")
tufts_local.mkdir(parents=True, exist_ok=True)
kaggle_tufts_candidates = [
    Path("/kaggle/input/datasets/rezanadimikj/tufts-panoramic"),
    Path("/kaggle/input/tufts-panoramic"),
    Path("/kaggle/input/tufts"),
]
tufts_mounted = False

if not FORCE_HF_DATASET_DOWNLOAD:
    for cand in kaggle_tufts_candidates:
        if cand.exists():
            target_img_dir = cand / "Radiographs" if (cand / "Radiographs").exists() else cand
            link_target = tufts_local / "Radiographs"
            if not link_target.exists():
                try:
                    link_target.symlink_to(target_img_dir)
                except Exception:
                    pass
            print(f"[DATASET] Attached Kaggle Tufts detected at {cand} (Skipping remote download).")
            tufts_mounted = True
            break

if not tufts_mounted:
    tufts_repo = os.environ.get("TUFTS_IMAGES_REPO", "Reza-Nadimi/tufts-train-images")
    has_tufts = (tufts_local / "Radiographs").exists() or (tufts_local / "images").exists() or Path("data/Tufts/Radiographs").exists()
    if tufts_repo and not has_tufts:
        print(f"\n[SYNC] Downloading Tufts panoramic images from Hugging Face ({tufts_repo})...")
        try:
            snapshot_download(
                repo_id=tufts_repo,
                repo_type="dataset",
                local_dir=str(tufts_local),
                token=hf_token,
            )
            print("[SYNC] Tufts images ready.")
        except Exception as e:
            print(f"[SYNC WARNING] Tufts images download failed: {e}")
    else:
        print("[SYNC] Tufts panoramic images verified on local disk.")


## 5. Interactive SFT Configuration & Curriculum Stage Selection



In [ ]:
from tabulate import tabulate

# =========================================================================
# STAGE 1 SFT EXECUTION PARAMETERS
# =========================================================================
# Curriculum Stage:
#   - 'dentex_alone'          : Stage 1a: DENTEX Alone + DENTEX Healthy Controls
#   - 'dentex_tufts_overlap'  : Stage 1b: DENTEX + Tufts Overlap + Negative Controls
#   - 'multicohort_all'       : Stage 1c: Full Multi-Cohort: DENTEX + Tufts All 4 Findings + Full Negative Controls
# =========================================================================
# BASE MODEL RESOLUTION: Attached Kaggle Input (Default) vs. Hugging Face Hub
# =========================================================================
# Path A (Default on Kaggle): Loads directly from attached private Kaggle input:
#   /kaggle/input/qwen3-5-9b (rezanadimikj/qwen3-5-9b)
#   -> 0 MB downloaded across the internet
#   -> 0 MB disk consumed from your 40.8 GB workspace quota
# Path B (Traditional / Auditor Fallback): Downloads directly from Hugging Face Hub:
#   'Qwen/Qwen3.5-9B'
#
# TOGGLE: Set FORCE_HF_MODEL_DOWNLOAD = True to force downloading from Hugging Face.
FORCE_HF_MODEL_DOWNLOAD = False

def resolve_base_model(default_repo="Qwen/Qwen3.5-9B") -> str:
    if not FORCE_HF_MODEL_DOWNLOAD:
        candidates = [
            Path("/kaggle/input/datasets/rezanadimikj/qwen3-5-9b"),
            Path("/kaggle/input/qwen3-5-9b"),
            Path("/kaggle/input/qwen3_5_9b"),
            Path("/kaggle/input/qwen-3-5-9b"),
            Path("/kaggle/input/qwen3-5-9b-base"),
        ]
        for cand in candidates:
            if cand.is_dir() and (cand / "config.json").exists():
                print(f"[RESOLVER] Detected attached Kaggle Base Model: {cand}")
                print("           (0 download, 0 disk space used from 40 GB workspace quota)")
                return str(cand)
    print(f"[RESOLVER] Using Hugging Face Hub model: {default_repo}")
    return default_repo

MODEL_ID = resolve_base_model()

STAGE = "dentex_alone"

# Training Track:
#   - 'with_tools' : Multi-turn diagnostic agent with 8 workstation tools
#   - 'no_tools'   : Direct radiologist (tool-free Chain-of-Thought)
TRACK = "with_tools"

# Vision LoRA Mode: 'projector' (merger.mlp.0, merger.mlp.2) or 'none'
LORA_TARGET_VISION = "projector"

# Precision: 'bf16' (native on Cloud TPU v5e-8 and Ampere+ GPUs), 'fp16', or 'qlora'
# NOTE: Keep as 'bf16'! When USE_FSDP=True, train_sft.py automatically loads master
# weights in float32 (~4.5 GB/core) for PyTorch/XLA FSDP, while running native BF16 compute.
PRECISION = "bf16"

# Hardware & Multi-Core Distribution (Cloud TPU v5e-8)
NUM_CORES = 8 if IS_TPU else 1
USE_FSDP = True if IS_TPU else False

# Hyperparameters
DATA_DIR = "data"
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 1 if (IS_TPU and NUM_CORES > 1) else 8
EPOCHS = 3
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

# Evaluation & Checkpointing Strategies
# Strategy options:
#   - 'epoch' : Run evaluation/checkpoint save at end of each completed epoch (Recommended & Default)
#   - 'steps' : Run evaluation/checkpoint save at step intervals (--eval-every-steps / --push-every-steps)
#   - 'both'  : Run at both step intervals and at epoch boundaries
EVAL_STRATEGY = "epoch"
EVAL_EVERY_STEPS = 0  # 0 = epoch-boundary only; > 0 = intermediate step evaluations

SAVE_STRATEGY = "epoch"
PUSH_EVERY_STEPS = 0  # 0 = epoch-boundary only; > 0 = intermediate step pushes to Hugging Face Hub

# Checkpoint Sync & Kaggle Continuity
HF_REPO = os.environ.get("HF_ARTIFACT_REPO", "Reza-Nadimi/vlm-dental-models")
RESUME = False

# Single Static Sequence Length (Zero Buckets across the project!)
# Primary target: 32768 (32k)
# Backtrack options if 32k hits an HBM limit on your TPU:
#   TARGET_SEQ_LEN = 24576  (24k)
#   TARGET_SEQ_LEN = 16384  (16k)
TARGET_SEQ_LEN = 32768

# Hardware Memory Wall Bypasses
XLA_PALLAS = True   # Enables TPU attention compilation and kernel optimizations (SDPA lowering)
XLA_SPMD = False    # Standard FSDP (False) vs GSPMD Sequence Sharding (True)

# XLA Persistent Compilation Cache (Auto-detected if attached as Kaggle input or in data/xla_cache)
def resolve_xla_cache_dir_notebook() -> str | None:
    def is_valid_cache(d: Path) -> bool:
        if not d.is_dir():
            return False
        # Valid cache must contain compilation artifacts other than just dataset-metadata.json
        files = [f for f in d.iterdir() if f.name != "dataset-metadata.json"]
        return len(files) > 0

    candidates = [
        Path("/kaggle/input/datasets/rezanadimikj/vlm-dental-xla-cache"),
        Path("/kaggle/input/datasets/rezanadimikj/vlm-dental-xla-cache/xla_cache"),
        Path("/kaggle/input/vlm-dental-xla-cache"),
        Path("/kaggle/input/vlm-dental-xla-cache/xla_cache"),
        Path("/kaggle/input/xla-cache-sft"),
        Path("/kaggle/working/xla_cache"),
        Path("data/xla_cache"),
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for item in sorted(kaggle_input.glob("**/datasets/*/*xla*cache*")):
            if item.is_dir():
                candidates.insert(0, item)
        for item in sorted(kaggle_input.glob("**/*xla*cache*")):
            if item.is_dir():
                candidates.insert(0, item)
        for item in sorted(kaggle_input.glob("*xla*cache*")):
            if item.is_dir():
                candidates.insert(0, item)

    for cand in candidates:
        target = cand / "xla_cache" if (cand / "xla_cache").is_dir() else cand
        if is_valid_cache(target):
            artifact_count = len(list(target.iterdir()))
            print(f"[XLA CACHE] Detected persistent compilation cache ({artifact_count} items): {target}")
            return str(target)

    print("[XLA CACHE] No persistent compilation cache detected. Training will compile on-the-fly.")
    return None

XLA_CACHE_DIR = resolve_xla_cache_dir_notebook()

# Optional Overrides (None uses canonical defaults per stage/track)
DATASET_PATH = None    # Path(s) to custom verified traces JSONL, or None for canonical
OUTPUT_DIR = None      # Custom output checkpoint dir, or None for default
MAX_SEQ_LEN = None     # Maximum token bucket ceiling (e.g. 32768), or None for auto

# Expected Checkpoint & Manifest Table
target_checkpoint = OUTPUT_DIR or f"data/models/qwen3_5_9b_sft_{TRACK}_{STAGE}"
hf_target_folder = f"sft/qwen3_5_9b_sft_{TRACK}_{STAGE}"

global_batch_size = BATCH_SIZE * GRAD_ACCUM_STEPS * NUM_CORES
approx_train_samples = 670
estimated_total_steps = int((approx_train_samples / NUM_CORES / GRAD_ACCUM_STEPS) * EPOCHS)
estimated_warmup_steps = max(int(estimated_total_steps * WARMUP_RATIO), 1)

manifest_data = [
    ["Curriculum Stage", STAGE],
    ["Training Track", TRACK],
    ["Base Model ID", MODEL_ID],
    ["Vision LoRA Adapter", f"{LORA_TARGET_VISION} (merger.mlp.0, merger.mlp.2)" if LORA_TARGET_VISION == "projector" else "none"],
    ["Native Resolutions", "Enabled (Zero pixel downsampling / clamping)"],
    ["Precision", PRECISION],
    ["Global Effective Batch Size", f"{global_batch_size} ({BATCH_SIZE} per core x {GRAD_ACCUM_STEPS} accum x {NUM_CORES} cores)"],
    ["Peak Learning Rate", f"{LEARNING_RATE} (Cosine with {int(WARMUP_RATIO*100)}% warmup, ~{estimated_warmup_steps} warmup steps)"],
    ["Estimated Total Steps", f"~{estimated_total_steps} optimization steps across {EPOCHS} epochs"],
    ["Gradient Clipping", f"max_norm = {MAX_GRAD_NORM}"],
    ["LoRA Config", f"r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}"],
    ["Static Sequence Length", f"{TARGET_SEQ_LEN} tokens (Zero buckets, single XLA graph)"],
    ["XLA Pallas FlashAttention", str(XLA_PALLAS)],
    ["XLA SPMD Sharding", str(XLA_SPMD)],
    ["XLA Persistent Cache", XLA_CACHE_DIR if XLA_CACHE_DIR else "None (On-The-Fly Compilation)"],
    ["Validation Strategy", f"{EVAL_STRATEGY} (eval-every-steps={EVAL_EVERY_STEPS})"],
    ["Checkpoint Strategy", f"{SAVE_STRATEGY} (push-every-steps={PUSH_EVERY_STEPS})"],
    ["Target Local Checkpoint", target_checkpoint],
    ["HF Models Repository", HF_REPO if HF_REPO else "Disabled"],
    ["HF Checkpoint Subfolder", hf_target_folder],
    ["TPU Multi-Core Topology", f"{NUM_CORES} cores (xmp.spawn)" if IS_TPU else "Single device (1 core/GPU)"],
    ["FSDP Parameter Sharding", f"Enabled (~4.5 GB FP32 base weights / core (BF16 compute))" if IS_TPU and USE_FSDP else "Disabled (DDP / Single)"],
    ["Resume from HF", str(RESUME)],
]

print("=" * 75)
print("VLM-DENTAL: STAGE 1 SFT EXECUTION MANIFEST")
print("=" * 75)
print(tabulate(manifest_data, headers=["Configuration Parameter", "Assigned Value"], tablefmt="fancy_grid"))


## 5b. Ahead-Of-Time (AOT) XLA Persistent Cache Warmup (Optional)

Pre-compiles the static sequence-length bucket computation graphs sequentially across the 8 Cloud TPU v5e-8 cores, saving compiled binaries to `/kaggle/working/xla_cache`.
Flushes compiler memory between buckets to prevent host CPU RAM spikes and eliminate the 15+ minute JIT compilation freeze during training.


In [ ]:
# =========================================================================
# AHEAD-OF-TIME (AOT) XLA COMPILATION WARMUP
# =========================================================================
# Set RUN_AOT_WARMUP = True to pre-compile the persistent cache on this VM.
# Compiles exactly ONE computation graph at TARGET_SEQ_LEN (default: 32768).
RUN_AOT_WARMUP = True
AOT_CACHE_DIR = "/kaggle/working/xla_cache"

if RUN_AOT_WARMUP:
    # Clear conflicting legacy TPU multi-host cluster address variables on Kaggle/Colab
    for _var in ["TPU_PROCESS_ADDRESSES", "TPU_PROCESS_COUNT", "CLOUD_TPU_TASK_ID", "PJRT_DEVICE", "XLA_FLAGS", "LIBTPU_INIT_ARGS"]:
        os.environ.pop(_var, None)

    warmup_cmd = [
        "python", "-W", "ignore::UserWarning", "scripts/warmup_xla_cache.py",
        "--model-id", MODEL_ID,
        "--cache-dir", AOT_CACHE_DIR,
        "--track", TRACK,
        "--precision", PRECISION,
        "--num-cores", str(NUM_CORES),
        "--max-seq-len", str(TARGET_SEQ_LEN),
        *(["--xla-spmd"] if XLA_SPMD else (["--fsdp"] if USE_FSDP else ["--no-fsdp"])),
        "--attn-implementation", "sdpa",
        "--lora-target-vision", LORA_TARGET_VISION,
    ]
    if XLA_PALLAS:
        warmup_cmd.append("--xla-pallas")
    else:
        warmup_cmd.append("--no-xla-pallas")

    warmup_cmd_str = " ".join(warmup_cmd)
    print(f"[LAUNCHING AOT WARMUP] Executing:\n{warmup_cmd_str}\n")
    # Clean up any lingering TPU device processes before launching
    !pkill -9 -f "multiprocessing.spawn" 2>/dev/null || true
    !pkill -9 -f "multiprocessing.resource_tracker" 2>/dev/null || true
    !pkill -9 -f "warmup_xla_cache.py" 2>/dev/null || true
    !pkill -9 -f "train_sft.py" 2>/dev/null || true
    !fuser -k -9 /dev/vfio/* 2>/dev/null || true
    !XLA_FLAGS="" LIBTPU_INIT_ARGS="" PYTHONWARNINGS="ignore::UserWarning" {warmup_cmd_str}
else:
    print("[INFO] RUN_AOT_WARMUP is False. Skipping pre-compilation (training will use attached cache or compile on-the-fly).")


## 5c. Export XLA Cache to Kaggle Dataset (One-Time Export)

Exports the pre-compiled `/kaggle/working/xla_cache` folder as a private Kaggle Dataset so it can be attached to future training notebooks for instant 0-second step 0 starts.


In [ ]:
# =========================================================================
# EXPORT XLA PERSISTENT CACHE TO KAGGLE DATASETS (VIA KAGGLE CLI)
# =========================================================================
# Set UPLOAD_TO_KAGGLE = True to publish the generated cache directory.
UPLOAD_TO_KAGGLE = True
KAGGLE_DATASET_ID = "rezanadimikj/vlm-dental-xla-cache"
CACHE_EXPORT_DIR = Path("/kaggle/working/xla_cache")

if UPLOAD_TO_KAGGLE and CACHE_EXPORT_DIR.is_dir():
    import json, os, sys
    from pathlib import Path

    # 1. Ensure dataset metadata exists for Kaggle CLI
    meta_file = CACHE_EXPORT_DIR / "dataset-metadata.json"
    if not meta_file.exists():
        meta = {
            "title": "VLM-DENTAL SFT XLA Persistent Cache",
            "id": KAGGLE_DATASET_ID,
            "licenses": [{"name": "apache-2.0"}],
            "description": "Pre-compiled XLA persistent cache for VLM-DENTAL SFT on Cloud TPU v5e-8.",
        }
        with open(meta_file, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)

    # 2. Upload via Kaggle CLI (dir-mode tar for rapid packaging)
    !{sys.executable} -m pip install -q kaggle
    print(f"[KAGGLE EXPORT] Uploading {CACHE_EXPORT_DIR} to {KAGGLE_DATASET_ID}...")
    !{sys.executable} -m kaggle datasets version -p {CACHE_EXPORT_DIR} -m "Update SFT XLA persistent cache" --dir-mode tar || {sys.executable} -m kaggle datasets create -p {CACHE_EXPORT_DIR} --dir-mode tar
elif not CACHE_EXPORT_DIR.is_dir():
    print(f"[INFO] Cache directory {CACHE_EXPORT_DIR} does not exist yet. Run Section 5b first.")
else:
    print("[INFO] UPLOAD_TO_KAGGLE is False. Set to True to push cache to Kaggle Datasets.")


## 6. Launch SFT Training Pipeline



In [ ]:
cmd = [
    "python", "-W", "ignore::UserWarning", "scripts/train_sft.py",
    "--track", TRACK,
    "--stage", STAGE,
    "--model-id", MODEL_ID,
    "--data-dir", DATA_DIR,
    "--lora-target-vision", LORA_TARGET_VISION,
    "--precision", PRECISION,
    "--batch-size", str(BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRAD_ACCUM_STEPS),
    "--learning-rate", str(LEARNING_RATE),
    "--warmup-ratio", str(WARMUP_RATIO),
    "--max-grad-norm", str(MAX_GRAD_NORM),
    "--epochs", str(EPOCHS),
    "--lora-r", str(LORA_R),
    "--lora-alpha", str(LORA_ALPHA),
    "--lora-dropout", str(LORA_DROPOUT),
    "--eval-strategy", EVAL_STRATEGY,
    "--eval-every-steps", str(EVAL_EVERY_STEPS),
    "--save-strategy", SAVE_STRATEGY,
    "--push-every-steps", str(PUSH_EVERY_STEPS),
    "--num-cores", str(NUM_CORES),
    "--max-seq-len", str(TARGET_SEQ_LEN),
    "--fsdp" if USE_FSDP else "--no-fsdp",
]

if XLA_PALLAS:
    cmd.append("--xla-pallas")
else:
    cmd.append("--no-xla-pallas")

if XLA_SPMD:
    cmd.append("--xla-spmd")

if XLA_CACHE_DIR:
    cmd.extend(["--xla-cache-dir", XLA_CACHE_DIR])

if DATASET_PATH:
    cmd.extend(["--dataset-path", DATASET_PATH])

if OUTPUT_DIR:
    cmd.extend(["--output-dir", OUTPUT_DIR])

if HF_REPO:
    cmd.extend(["--hf-repo", HF_REPO])

if RESUME and HF_REPO:
    cmd.extend(["--resume-hf", HF_REPO])

# Clear conflicting legacy TPU multi-host cluster address variables on Kaggle/Colab
for _var in ["TPU_PROCESS_ADDRESSES", "TPU_PROCESS_COUNT", "CLOUD_TPU_TASK_ID", "PJRT_DEVICE"]:
    os.environ.pop(_var, None)

cmd_str = " ".join(cmd)
print(f"[LAUNCHING PIPELINE] Executing:\n{cmd_str}\n")
# Clean up any lingering TPU device processes before launching
!pkill -9 -f "multiprocessing.spawn" 2>/dev/null || true
!pkill -9 -f "multiprocessing.resource_tracker" 2>/dev/null || true
!pkill -9 -f "train_sft.py" 2>/dev/null || true
!pkill -9 -f "run_grpo.py" 2>/dev/null || true
!fuser -k -9 /dev/vfio/* 2>/dev/null || true
    !XLA_FLAGS="" LIBTPU_INIT_ARGS="" PYTHONWARNINGS="ignore::UserWarning" {cmd_str}


## 7. Training Loss & Convergence Visualizer

Plots conversational assistant training loss and held-out validation loss curves, reporting initial, best validation, and final loss metrics.

In [ ]:
import json
import matplotlib.pyplot as plt

output_dir = f"data/models/qwen3_5_9b_sft_{TRACK}_{STAGE}"
log_file = f"{output_dir}/training_loss.jsonl"

if os.path.exists(log_file):
    steps, train_losses, val_steps, val_losses = [], [], [], []
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                step = rec.get("step", len(steps))
                steps.append(step)
                train_losses.append(rec.get("loss", 0.0))
                if "val_loss" in rec:
                    val_steps.append(step)
                    val_losses.append(rec["val_loss"])

    plt.figure(figsize=(11, 5))
    plt.plot(steps, train_losses, label="Assistant Training Loss", color="#1f77b4", lw=1.8, alpha=0.85)
    if val_losses:
        plt.plot(val_steps, val_losses, label="Validation Loss (5% held-out)", color="#d62728", marker="o", lw=2)

    plt.title(f"VLM-DENTAL Stage 1 SFT Convergence — Stage: {STAGE.upper()} | Track: {TRACK.upper()}")
    plt.xlabel("Optimization Steps")
    plt.ylabel("Assistant Conversational Cross-Entropy Loss")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    best_val_str = f" | Best Val Loss: {min(val_losses):.4f}" if val_losses else ""
    print(f"[METRICS] Initial Loss: {train_losses[0]:.4f} -> Final Loss: {train_losses[-1]:.4f}{best_val_str}")
    print(f"[CHECKPOINTS] Checkpoints saved to {output_dir}")
else:
    print(f"[INFO] Log file {log_file} not found yet. Execute Cell 6 to run training.")

## 8. Multi-Condition Evaluation on DENTEX Test Split (§6, §20, §26)

Unified evaluation across baseline and fine-tuned models on the official DENTEX test split using `scripts/evaluate_models.py`.
Supports all conditions (`base_no_tools`, `base_with_tools`, `sft_no_tools`, `sft_with_tools`, or `all`), set-level multi-finding matching (Rule 13), and cached zero-shot baseline re-use.


In [ ]:
# =========================================================================
# MULTI-CONDITION EVALUATION PARAMETERS (§6, §20, §26)
# =========================================================================
# Condition choices:
#   - 'sft_with_tools' : Fine-tuned agent with 8 workstation tools
#   - 'sft_no_tools'   : Fine-tuned direct radiologist (tool-free CoT)
#   - 'base_with_tools': Untuned base model with 8 workstation tools
#   - 'base_no_tools'  : Untuned base model direct zero-shot
#   - 'all'            : Sequentially evaluate all conditions
EVAL_CONDITION = f"sft_{TRACK}"
EVAL_DATASET = "dentex"
EVAL_SPLIT = "test"
EVAL_ADAPTER_PATH = target_checkpoint  # Evaluates the checkpoint just trained
REUSE_CACHED_BASE = True               # Reuses cached zero-shot baseline for base_no_tools
EVAL_LIMIT = None                      # None for full split, or int (e.g. 5) for quick smoke test
EVAL_MAX_TURNS = 25
EVAL_MAX_TOOL_CALLS = 50
EVAL_PRECISION = "bf16"

eval_cmd = [
    "python", "scripts/evaluate_models.py",
    "--condition", EVAL_CONDITION,
    "--model-id", MODEL_ID,
    "--dataset", EVAL_DATASET,
    "--split", EVAL_SPLIT,
    "--data-dir", DATA_DIR,
    "--output-dir", "data/evaluations",
    "--precision", EVAL_PRECISION,
    "--max-turns", str(EVAL_MAX_TURNS),
    "--max-tool-calls", str(EVAL_MAX_TOOL_CALLS),
]

if EVAL_ADAPTER_PATH and not EVAL_CONDITION.startswith("base_"):
    eval_cmd.extend(["--adapter-path", EVAL_ADAPTER_PATH])

if REUSE_CACHED_BASE:
    eval_cmd.append("--reuse-cached")

if EVAL_LIMIT is not None:
    eval_cmd.extend(["--limit", str(EVAL_LIMIT)])

eval_cmd_str = " ".join(eval_cmd)
print(f"[LAUNCHING EVALUATION] Executing:\n{eval_cmd_str}\n")
!{eval_cmd_str}
